In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, make_union, Pipeline
import xgboost as xgb
import joblib
import os
import warnings
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from rdkit.Chem import Draw, AllChem
import argparse

warnings.filterwarnings('ignore')

from skfp.fingerprints import (
    PubChemFingerprint,
    KlekotaRothFingerprint,
    ECFPFingerprint,
    MACCSFingerprint,
    AtomPairFingerprint,
    RDKitFingerprint,
)
from skfp.preprocessing import MolFromSmilesTransformer

# ==================== Checkpoint Management ====================
def save_checkpoint(fp_name, model_name, results, checkpoint_dir='checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_file = f"{checkpoint_dir}/{fp_name.replace(' ', '_')}_{model_name}.pkl"
    joblib.dump(results, checkpoint_file)
    print(f"✓ Checkpoint saved: {checkpoint_file}")

def load_checkpoint(fp_name, model_name, checkpoint_dir='checkpoints'):
    checkpoint_file = f"{checkpoint_dir}/{fp_name.replace(' ', '_')}_{model_name}.pkl"
    if os.path.exists(checkpoint_file):
        return joblib.load(checkpoint_file)
    return None

def is_model_trained(fp_name, model_name, checkpoint_dir='checkpoints'):
    checkpoint_file = f"{checkpoint_dir}/{fp_name.replace(' ', '_')}_{model_name}.pkl"
    return os.path.exists(checkpoint_file)

def save_all_results(all_results, filename='all_results.pkl'):
    joblib.dump(all_results, filename)
    print(f"✓ All results saved to: {filename}")

def load_all_results(filename='all_results.pkl'):
    if os.path.exists(filename):
        return joblib.load(filename)
    return {}

# ==================== Fingerprint Pipeline ====================
def create_fingerprint_pipeline(fp_type='pubchem'):
    fp_map = {
        'pubchem': PubChemFingerprint(),
        'klekotaroth': KlekotaRothFingerprint(),
        'ecfp': ECFPFingerprint(radius=2, fp_size=2048),
        'maccs': MACCSFingerprint(),
        'atompairs': AtomPairFingerprint(fp_size=2048),
        'rdkit': RDKitFingerprint(fp_size=2048),
    }
    if fp_type not in fp_map:
        raise ValueError(f"Unsupported fingerprint type: {fp_type}")
    fp_info = {
        'pubchem': '881 bits',
        'klekotaroth': '4860 bits',
        'ecfp': '2048 bits (ECFP4/Morgan)',
        'maccs': '167 bits',
        'atompairs': '2048 bits',
        'rdkit': '2048 bits (RDKit/FP2)',
    }
    print(f"  Generating fingerprint: {fp_type} - {fp_info.get(fp_type, '')}")
    return make_pipeline(
        MolFromSmilesTransformer(),
        fp_map[fp_type]
    )

def create_multi_fp_union(fp_types=None):
    fp_map = {
        'pubchem': PubChemFingerprint(),
        'klekotaroth': KlekotaRothFingerprint(),
        'ecfp': ECFPFingerprint(radius=2, fp_size=2048),
        'maccs': MACCSFingerprint(),
        'atompairs': AtomPairFingerprint(fp_size=2048),
        'rdkit': RDKitFingerprint(fp_size=2048),
    }
    fp_info = {
        'pubchem': 881,
        'klekotaroth': 4860,
        'ecfp': 2048,
        'maccs': 167,
        'atompairs': 2048,
        'rdkit': 2048,
    }
    fingerprinters = []
    fp_names = []
    total_bits = 0
    for fp_type in fp_types:
        if fp_type in fp_map:
            fingerprinters.append(fp_map[fp_type])
            fp_names.append(fp_type)
            total_bits += fp_info.get(fp_type, 0)
        else:
            print(f"Warning: Unsupported fingerprint type {fp_type}, skipping")
    if not fingerprinters:
        raise ValueError("No valid fingerprint types available")
    fp_name_str = '+'.join([name[:3] for name in fp_names])
    print(f"  Union fingerprint: {', '.join(fp_names)}")
    print(f"  Total bits: {total_bits} bits")
    return make_pipeline(
        MolFromSmilesTransformer(),
        make_union(*fingerprinters)
    ), fp_name_str, total_bits

def create_model_pipeline(fp_type, model, scaler=False):
    fp_pipeline = create_fingerprint_pipeline(fp_type)
    if scaler:
        return Pipeline([
            ('fingerprints', fp_pipeline),
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])
    else:
        return Pipeline([
            ('fingerprints', fp_pipeline),
            ('classifier', model)
        ])

# ==================== Evaluation Metrics ====================
def calculate_metrics(y_true, y_pred, y_pred_proba):
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    se = tp / (tp + fn) if (tp + fn) > 0 else 0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    ba = (se + sp) / 2
    return {
        'auc': auc_roc, 
        'accuracy': accuracy, 
        'f1': f1, 
        'mcc': mcc,
        'se': se, 
        'sp': sp, 
        'ba': ba, 
        'precision': precision, 
        'npv': npv, 
        'tp': int(tp), 
        'tn': int(tn), 
        'fp': int(fp), 
        'fn': int(fn)
    }

def print_metrics(results, model_name, fp_name):
    print(f"\n{'='*50}")
    print(f"{model_name} with {fp_name} - Test Set Evaluation Results")
    print(f"{'='*50}")
    print(f"Confusion Matrix:")
    print(f"  TP: {results['tp']:4d}  FN: {results['fn']:4d}")
    print(f"  FP: {results['fp']:4d}  TN: {results['tn']:4d}")
    print(f"\nPerformance Metrics:")
    print(f"  Sensitivity (SE):          {results['se']:.4f}")
    print(f"  Specificity (SP):          {results['sp']:.4f}")
    print(f"  Balanced Accuracy (BA):    {results['ba']:.4f}")
    print(f"  Precision:                 {results['precision']:.4f}")
    print(f"  Negative Predictive Value: {results['npv']:.4f}")
    print(f"  Accuracy (Q):              {results['accuracy']:.4f}")
    print(f"  F1 Score:                  {results['f1']:.4f}")
    print(f"  MCC:                       {results['mcc']:.4f}")
    print(f"  AUC:                       {results['auc']:.4f}")

# ==================== Base Model Training ====================
def train_knn(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"KNN with {fp_name}")
    print(f"{'='*60}")
    model = KNeighborsClassifier()
    param_grid = {
        'classifier__n_neighbors': [3, 5, 7, 10, 15],
        'classifier__weights': ['uniform', 'distance'],
        'classifier__metric': ['minkowski', 'euclidean', 'manhattan']
    }
    pipeline = create_model_pipeline(fp_type, model, scaler=False)
    print("\nStarting KNN hyperparameter optimization...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', 
                              n_jobs=-1, verbose=1)
    grid_search.fit(smiles_train, y_train)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV AUC: {grid_search.best_score_:.4f}")
    y_pred_proba = grid_search.predict_proba(smiles_test)[:, 1]
    y_pred = grid_search.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "KNN", fp_name)
    return {
        'model': grid_search.best_estimator_,
        'results': results,
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_
    }

def train_svm(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"SVM with {fp_name}")
    print(f"{'='*60}")
    model = SVC(probability=True, random_state=42)
    param_grid = {
        'scaler': [StandardScaler(), None],
        'classifier__C': [0.1, 1, 10, 100],
        'classifier__gamma': ['scale', 'auto', 0.1, 0.01],
        'classifier__kernel': ['rbf', 'linear']
    }
    pipeline = Pipeline([
        ('fingerprints', create_fingerprint_pipeline(fp_type)),
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    print("\nStarting SVM hyperparameter optimization...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', 
                              n_jobs=-1, verbose=1)
    grid_search.fit(smiles_train, y_train)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV AUC: {grid_search.best_score_:.4f}")
    y_pred_proba = grid_search.predict_proba(smiles_test)[:, 1]
    y_pred = grid_search.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "SVM", fp_name)
    return {
        'model': grid_search.best_estimator_,
        'results': results,
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_
    }

def train_random_forest(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"Random Forest with {fp_name}")
    print(f"{'='*60}")
    model = RandomForestClassifier(random_state=42)
    param_grid = {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__max_depth': [5, 10, 20, 30, None],
        'classifier__min_samples_leaf': [1, 10],
        'classifier__max_features': ['sqrt', 'log2']
    }
    pipeline = create_model_pipeline(fp_type, model, scaler=False)
    print("\nStarting Random Forest hyperparameter optimization...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', 
                              n_jobs=-1, verbose=1)
    grid_search.fit(smiles_train, y_train)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV AUC: {grid_search.best_score_:.4f}")
    y_pred_proba = grid_search.predict_proba(smiles_test)[:, 1]
    y_pred = grid_search.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "Random Forest", fp_name)
    return {
        'model': grid_search.best_estimator_,
        'results': results,
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_
    }

def train_xgboost(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"XGBoost with {fp_name}")
    print(f"{'='*60}")
    model = xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
    param_grid = {
        'classifier__learning_rate': [0.01, 0.05, 0.1],
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 5, 7, 10],
        'classifier__min_child_weight': [1, 2, 3],
        'classifier__subsample': [0.8, 0.9, 1.0],
        'classifier__colsample_bytree': [0.8, 0.9, 1.0]
    }
    pipeline = create_model_pipeline(fp_type, model, scaler=False)
    print("\nStarting XGBoost hyperparameter optimization...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', 
                              n_jobs=-1, verbose=1)
    grid_search.fit(smiles_train, y_train)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV AUC: {grid_search.best_score_:.4f}")
    y_pred_proba = grid_search.predict_proba(smiles_test)[:, 1]
    y_pred = grid_search.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "XGBoost", fp_name)
    return {
        'model': grid_search.best_estimator_,
        'results': results,
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_
    }

def train_gaussian_nb(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"Gaussian NB with {fp_name}")
    print(f"{'='*60}")
    fp_pipeline = create_fingerprint_pipeline(fp_type)
    X_train = fp_pipeline.fit_transform(smiles_train)
    X_test = fp_pipeline.transform(smiles_test)
    X_train = X_train.astype(float)
    X_test = X_test.astype(float)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    gnb = GaussianNB()
    gnb.fit(X_train_scaled, y_train)
    y_pred = gnb.predict(X_test_scaled)
    y_pred_proba = gnb.predict_proba(X_test_scaled)[:, 1]
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "Gaussian NB", fp_name)
    return {
        'model': gnb,
        'scaler': scaler,
        'fp_pipeline': fp_pipeline,
        'results': results,
        'best_params': {},
        'cv_score': results['auc']
    }

# ==================== Single Fingerprint Ensemble Models ====================
def train_voting_ensemble(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"Voting Ensemble with {fp_name}")
    print(f"{'='*60}")
    fp_pipeline = create_fingerprint_pipeline(fp_type)
    X_train = fp_pipeline.fit_transform(smiles_train)
    X_test = fp_pipeline.transform(smiles_test)
    X_train = X_train.astype(float)
    X_test = X_test.astype(float)
    rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
    xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, 
                                 random_state=42, eval_metric='logloss', use_label_encoder=False)
    knn = KNeighborsClassifier(n_neighbors=7, weights='distance')
    estimators = [('rf', rf), ('xgb', xgb_model), ('knn', knn)]
    best_auc = 0
    best_voting = None
    best_strategy = 'soft'
    for strategy in ['soft', 'hard']:
        try:
            voting_clf = VotingClassifier(estimators=estimators, voting=strategy)
            voting_clf.fit(X_train, y_train)
            y_pred_proba = voting_clf.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_pred_proba)
            if auc > best_auc:
                best_auc = auc
                best_voting = voting_clf
                best_strategy = strategy
        except Exception as e:
            continue
    final_voting = VotingClassifier(estimators=estimators, voting=best_strategy)
    final_voting.fit(X_train, y_train)
    y_pred_proba = final_voting.predict_proba(X_test)[:, 1]
    y_pred = final_voting.predict(X_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print(f"\nBest Voting Ensemble Configuration:")
    print(f"  Strategy: {best_strategy}")
    print(f"  AUC: {best_auc:.4f}")
    print_metrics(results, "Voting Ensemble", fp_name)
    return {
        'model': final_voting,
        'fp_pipeline': fp_pipeline,
        'results': results,
        'best_params': {'voting': best_strategy},
        'cv_score': best_auc
    }

def train_stacking_ensemble(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type):
    print(f"\n{'='*60}")
    print(f"Stacking Ensemble with {fp_name}")
    print(f"{'='*60}")
    fp_pipeline = create_fingerprint_pipeline(fp_type)
    X_train = fp_pipeline.fit_transform(smiles_train)
    X_test = fp_pipeline.transform(smiles_test)
    X_train = X_train.astype(float)
    X_test = X_test.astype(float)
    estimators = [
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', xgb.XGBClassifier(n_estimators=100, random_state=42, 
                                 eval_metric='logloss', use_label_encoder=False)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ]
    final_clf = LogisticRegression(max_iter=1000, random_state=42)
    stacking_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=final_clf,
        cv=5,
        stack_method='predict_proba'
    )
    stacking_clf.fit(X_train, y_train)
    y_pred_proba = stacking_clf.predict_proba(X_test)[:, 1]
    y_pred = stacking_clf.predict(X_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "Stacking Ensemble", fp_name)
    return {
        'model': stacking_clf,
        'fp_pipeline': fp_pipeline,
        'results': results,
        'best_params': {'final_estimator': 'LogisticRegression'},
        'cv_score': results['auc']
    }

# ==================== Multi-Fingerprint Ensemble Models ====================
def train_multi_fp_union_pipeline(smiles_train, smiles_test, y_train, y_test, fp_types, combo_name):
    fp_names = [fp_type_to_name[ft] for ft in fp_types]
    fp_display_name = f"{len(fp_types)}-FP: {'+'.join([n[:3] for n in fp_names])}"
    print(f"\n{'='*60}")
    print(f"Multi-Fingerprint Union Pipeline - {combo_name}")
    print(f"{'='*60}")
    multi_fp_pipeline, fp_name_str, total_bits = create_multi_fp_union(fp_types)
    pipeline = Pipeline([
        ('multi_fp', multi_fp_pipeline),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20, None],
        'classifier__min_samples_leaf': [1, 5]
    }
    print("\nStarting multi-fingerprint union model hyperparameter optimization...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', 
                              n_jobs=-1, verbose=1)
    grid_search.fit(smiles_train, y_train)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV AUC: {grid_search.best_score_:.4f}")
    y_pred_proba = grid_search.predict_proba(smiles_test)[:, 1]
    y_pred = grid_search.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "Multi-FP Union", fp_display_name)
    return {
        'model': grid_search.best_estimator_,
        'results': results,
        'best_params': {**grid_search.best_params_, 'fp_types': fp_types, 'fp_names': fp_names},
        'cv_score': grid_search.best_score_,
        'fp_types': fp_types,
        'total_bits': total_bits
    }

def train_multi_fp_voting(smiles_train, smiles_test, y_train, y_test, fp_types, combo_name):
    fp_names = [fp_type_to_name[ft] for ft in fp_types]
    fp_display_name = f"{len(fp_types)}-FP: {'+'.join([n[:3] for n in fp_names])}"
    print(f"\n{'='*60}")
    print(f"Multi-Fingerprint Voting Ensemble - {combo_name}")
    print(f"{'='*60}")
    models = {}
    predictions = {}
    aucs = {}
    for fp_type in fp_types:
        fp_name = fp_type_to_name[fp_type]
        print(f"\nTraining XGBoost model for {fp_name} fingerprint...")
        fp_pipeline = create_fingerprint_pipeline(fp_type)
        X_train = fp_pipeline.fit_transform(smiles_train)
        X_test = fp_pipeline.transform(smiles_test)
        X_train = X_train.astype(float)
        X_test = X_test.astype(float)
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False
        )
        model.fit(X_train, y_train)
        models[fp_type] = {'model': model, 'pipeline': fp_pipeline}
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_pred_proba)
        predictions[fp_type] = y_pred_proba
        aucs[fp_type] = auc
        print(f"  {fp_name} AUC: {auc:.4f}")
    total_auc = sum(aucs.values())
    weights = {fp: auc/total_auc for fp, auc in aucs.items()}
    weighted_proba = np.zeros_like(list(predictions.values())[0])
    for fp_type, pred in predictions.items():
        weighted_proba += weights[fp_type] * pred
    avg_proba = np.mean(list(predictions.values()), axis=0)
    y_pred_weighted = (weighted_proba >= 0.5).astype(int)
    results_weighted = calculate_metrics(y_test, y_pred_weighted, weighted_proba)
    y_pred_avg = (avg_proba >= 0.5).astype(int)
    results_avg = calculate_metrics(y_test, y_pred_avg, avg_proba)
    print(f"\nWeighted Voting (AUC-based weights):")
    for fp_type, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        fp_name = fp_type_to_name[fp_type]
        print(f"  {fp_name}: {weight:.3f}")
    print(f"  AUC: {results_weighted['auc']:.4f}")
    print(f"\nAverage Voting:")
    print(f"  AUC: {results_avg['auc']:.4f}")
    if results_weighted['auc'] >= results_avg['auc']:
        best_results = results_weighted
        best_method = 'weighted'
    else:
        best_results = results_avg
        best_method = 'average'
    print_metrics(best_results, "Multi-FP Voting", fp_display_name)
    return {
        'model': models,
        'results': best_results,
        'best_params': {
            'voting_method': best_method,
            'weights': weights if best_method == 'weighted' else None,
            'individual_aucs': aucs,
            'fp_types': fp_types,
            'fp_names': fp_names
        },
        'cv_score': best_results['auc'],
        'fp_types': fp_types
    }

def train_multi_fp_stacking(smiles_train, smiles_test, y_train, y_test, fp_types, combo_name):
    fp_names = [fp_type_to_name[ft] for ft in fp_types]
    fp_display_name = f"{len(fp_types)}-FP: {'+'.join([n[:3] for n in fp_names])}"
    print(f"\n{'='*60}")
    print(f"Multi-Fingerprint Stacking Ensemble - {combo_name}")
    print(f"{'='*60}")
    base_models = []
    for fp_type in fp_types:
        fp_name = fp_type_to_name[fp_type]
        print(f"\nTraining base model for {fp_name} fingerprint...")
        fp_pipeline = create_fingerprint_pipeline(fp_type)
        X_train = fp_pipeline.fit_transform(smiles_train)
        X_test = fp_pipeline.transform(smiles_test)
        X_train = X_train.astype(float)
        X_test = X_test.astype(float)
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        base_models.append((f'fp_{fp_type}', 
                           Pipeline([
                               ('fingerprints', fp_pipeline),
                               ('classifier', model)
                           ])))
    meta_learner = LogisticRegression(max_iter=1000, random_state=42)
    stacking_clf = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_learner,
        cv=5,
        stack_method='predict_proba'
    )
    stacking_clf.fit(smiles_train, y_train)
    y_pred_proba = stacking_clf.predict_proba(smiles_test)[:, 1]
    y_pred = stacking_clf.predict(smiles_test)
    results = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_metrics(results, "Multi-FP Stacking", fp_display_name)
    return {
        'model': stacking_clf,
        'results': results,
        'best_params': {'fp_types': fp_types, 'fp_names': fp_names},
        'cv_score': results['auc'],
        'fp_types': fp_types
    }

# ==================== Prediction Function ====================
def predict_activity(smiles_list, model_obj, model_type='single'):
    if isinstance(smiles_list, str):
        smiles_list = [smiles_list]
    if model_type == 'multi_fp_voting':
        predictions_list = []
        for fp_type, fp_model_dict in model_obj.items():
            if isinstance(fp_model_dict, dict) and 'pipeline' in fp_model_dict and 'model' in fp_model_dict:
                pipeline = fp_model_dict['pipeline']
                model = fp_model_dict['model']
                X = pipeline.transform(smiles_list)
                X = X.astype(float)
                proba = model.predict_proba(X)[:, 1]
                predictions_list.append(proba)
        if predictions_list:
            final_probabilities = np.mean(predictions_list, axis=0)
            final_predictions = (final_probabilities >= 0.5).astype(int)
        else:
            final_probabilities = np.array([0.5] * len(smiles_list))
            final_predictions = np.array([0] * len(smiles_list))
    elif model_type == 'pipeline' or 'StackingClassifier' in str(type(model_obj)):
        try:
            final_probabilities = model_obj.predict_proba(smiles_list)[:, 1]
            final_predictions = model_obj.predict(smiles_list)
        except:
            final_probabilities = np.array([0.5] * len(smiles_list))
            final_predictions = np.array([0] * len(smiles_list))
    else:
        try:
            fp_pipeline = model_obj.get('fp_pipeline')
            classifier = model_obj.get('model')
            scaler = model_obj.get('scaler')
            if fp_pipeline is not None and classifier is not None:
                X = fp_pipeline.transform(smiles_list)
                X = X.astype(float)
                if scaler:
                    X = scaler.transform(X)
                final_probabilities = classifier.predict_proba(X)[:, 1]
                final_predictions = classifier.predict(X)
            else:
                final_probabilities = model_obj.predict_proba(smiles_list)[:, 1]
                final_predictions = model_obj.predict(smiles_list)
        except:
            final_probabilities = np.array([0.5] * len(smiles_list))
            final_predictions = np.array([0] * len(smiles_list))
    return {
        'smiles': smiles_list,
        'predictions': final_predictions,
        'probabilities': final_probabilities,
        'activity': ['Active' if p == 1 else 'Inactive' for p in final_predictions]
    }

# ==================== Fingerprint Type Mapping and Combination Generation ====================
fp_type_to_name = {
    'pubchem': 'PubChem',
    'klekotaroth': 'Klekota-Roth',
    'ecfp': 'ECFP',
    'maccs': 'MACCS',
    'atompairs': 'AtomPairs',
    'rdkit': 'RDKit'
}
fp_name_to_type = {v: k for k, v in fp_type_to_name.items()}
all_fp_types = ['pubchem', 'klekotaroth', 'ecfp', 'maccs', 'atompairs', 'rdkit']

def generate_all_fp_combinations():
    combinations = {}
    combinations['2'] = list(itertools.combinations(all_fp_types, 2))
    combinations['3'] = list(itertools.combinations(all_fp_types, 3))
    combinations['4'] = list(itertools.combinations(all_fp_types, 4))
    combinations['5'] = list(itertools.combinations(all_fp_types, 5))
    combinations['6'] = [tuple(all_fp_types)]
    return combinations

# ==================== Plotting Functions ====================
def plot_all_figures(df, all_results, y_test, smiles_test, X_test_feats, fingerprint_configs, fp_combinations):
    sns.set_style('whitegrid')
    plt.rcParams['font.size'] = 12
    plt.rcParams['figure.dpi'] = 150
    output_dir = 'figures'
    os.makedirs(output_dir, exist_ok=True)

    # Fig.2: Descriptor distribution
    desc_cols = ['molecular_weight', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'MolLogP', 'MolMR']
    titles = ['Molecular weight', 'TPSA', 'H-bond donors', 'H-bond acceptors', 'LogP', 'Molar refractivity']
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for i, col in enumerate(desc_cols):
        ax = axes[i]
        toxic = df[df['LABEL']==1][col].dropna()
        nontoxic = df[df['LABEL']==0][col].dropna()
        sns.kdeplot(toxic, ax=ax, label='Toxic', color='blue', shade=True)
        sns.kdeplot(nontoxic, ax=ax, label='Non-toxic', color='orange', shade=True)
        ax.set_title(titles[i])
        ax.set_xlabel(col)
        ax.set_ylabel('Density')
        ax.legend()
    plt.tight_layout()
    plt.savefig(f'{output_dir}/Fig2_descriptor_distribution.pdf', dpi=300)
    plt.close()

    # Fig.3: ROC curves
    model_groups = {
        'XGBoost': ['XGBoost_pubchem', 'XGBoost_klekotaroth', 'XGBoost_ecfp', 'XGBoost_maccs', 'XGBoost_atompairs', 'XGBoost_rdkit'],
        'RandomForest': ['RandomForest_pubchem', 'RandomForest_klekotaroth', 'RandomForest_ecfp', 'RandomForest_maccs', 'RandomForest_atompairs', 'RandomForest_rdkit'],
        'SVM': ['SVM_pubchem', 'SVM_klekotaroth', 'SVM_ecfp', 'SVM_maccs', 'SVM_atompairs', 'SVM_rdkit'],
        'KNN': ['KNN_pubchem', 'KNN_klekotaroth', 'KNN_ecfp', 'KNN_maccs', 'KNN_atompairs', 'KNN_rdkit'],
        'GaussianNB': ['GaussianNB_pubchem', 'GaussianNB_klekotaroth', 'GaussianNB_ecfp', 'GaussianNB_maccs', 'GaussianNB_atompairs', 'GaussianNB_rdkit']
    }
    fp_display = {
        'pubchem': 'PubChem',
        'klekotaroth': 'Klekota-Roth',
        'ecfp': 'Morgan',
        'maccs': 'MACCS',
        'atompairs': 'AtomPairs',
        'rdkit': 'RDKit'
    }
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    ax_idx = 0
    for model_name, keys in model_groups.items():
        ax = axes[ax_idx]
        ax.plot([0,1], [0,1], 'k--', lw=1)
        for key in keys:
            if key not in all_results:
                continue
            model_info = all_results[key]
            fp = key.split('_')[1]

            y_proba = None
            if 'fp_pipeline' in model_info and model_info['fp_pipeline'] is not None:
                fp_pipeline = model_info['fp_pipeline']
                X_test_fp = fp_pipeline.transform(smiles_test)
                X_test_fp = X_test_fp.astype(float)
                if 'scaler' in model_info and model_info['scaler'] is not None:
                    X_test_fp = model_info['scaler'].transform(X_test_fp)
                model = model_info['model']
                y_proba = model.predict_proba(X_test_fp)[:, 1]
            elif 'model' in model_info and model_info['model'] is not None:
                model = model_info['model']
                try:
                    y_proba = model.predict_proba(smiles_test)[:, 1]
                except Exception as e:
                    print(f"  Warning: Model {key} prediction failed: {e}")
                    continue
            else:
                print(f"  Warning: Model {key} has no valid prediction method, skipping")
                continue

            if y_proba is None:
                continue

            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = roc_auc_score(y_test, y_proba)
            ax.plot(fpr, tpr, lw=2, label=f"{fp_display[fp]} (AUC={roc_auc:.2f})")

        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(model_name)
        ax.legend(loc='lower right')
        ax_idx += 1
    axes[5].axis('off')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/Fig3_ROC_curves.pdf', dpi=300)
    plt.close()

    # Fig.4A: Performance heatmap
    selected_models = []
    for base in ['XGBoost', 'RandomForest', 'SVM', 'KNN', 'GaussianNB']:
        for fp in ['ecfp', 'maccs']:
            key = f'{base}_{fp}'
            if key in all_results:
                selected_models.append(key)
    for key in all_results:
        if 'Voting' in key or 'Stacking' in key or 'MultiFP' in key:
            selected_models.append(key)
            if len(selected_models) >= 20:
                break
    metrics_list = ['auc', 'accuracy', 'f1', 'mcc', 'se', 'sp', 'ba']
    data_heat = []
    model_labels = []
    for mod in selected_models:
        if mod in all_results and 'results' in all_results[mod]:
            row = [all_results[mod]['results'][m] for m in metrics_list]
            data_heat.append(row)
            model_labels.append(mod[:20])
    if data_heat:
        df_heat = pd.DataFrame(data_heat, index=model_labels, columns=[m.upper() for m in metrics_list])
        plt.figure(figsize=(12, max(6, len(model_labels)*0.4)))
        sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='viridis', linewidths=0.5)
        plt.title('Performance Metrics of Selected Models')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/Fig4A_heatmap.pdf', dpi=300)
        plt.close()

    # Fig.4B: Factorial ANOVA
    try:
        anova_data = []
        for model in ['XGBoost_ecfp', 'SVM_ecfp']:
            if model in all_results and 'cv_scores' in all_results[model]:
                scores = all_results[model]['cv_scores']
                for fold, score in enumerate(scores):
                    anova_data.append({'Model': model, 'Fold': fold, 'AUC': score})
        if anova_data:
            df_anova = pd.DataFrame(anova_data)
            plt.figure(figsize=(8,6))
            sns.barplot(x='Model', y='AUC', data=df_anova, capsize=0.1, errwidth=1.5)
            plt.title('Factorial ANOVA: AUC by Model (5-fold CV)')
            plt.ylabel('AUC')
            plt.ylim(0.8, 1.0)
            plt.tight_layout()
            plt.savefig(f'{output_dir}/Fig4B_ANOVA.pdf', dpi=300)
            plt.close()
    except:
        pass

    # Fig.5: Ensemble model comparison
    ensemble_models = {}
    for key in all_results:
        if 'Voting' in key or 'Stacking' in key or 'MultiFP' in key:
            if 'results' in all_results[key]:
                ensemble_models[key] = all_results[key]['results']
    if ensemble_models:
        metrics_plot = ['accuracy', 'f1', 'ba', 'auc']
        x = np.arange(len(metrics_plot))
        width = 0.15
        fig, ax = plt.subplots(figsize=(12, 6))
        for i, (name, res) in enumerate(ensemble_models.items()):
            values = [res[m] for m in metrics_plot]
            ax.bar(x + i*width, values, width, label=name[:20])
        ax.set_xticks(x + width*(len(ensemble_models)-1)/2)
        ax.set_xticklabels([m.upper() for m in metrics_plot])
        ax.set_ylabel('Score')
        ax.set_ylim(0.5, 1.0)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.set_title('Ensemble Models Performance')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/Fig5_ensemble_comparison.pdf', dpi=300)
        plt.close()

    # Fig.6: Best model confusion matrix
    best_key = max(all_results.keys(), key=lambda k: all_results[k]['results']['auc'] if 'results' in all_results[k] else 0)
    if best_key in all_results and 'results' in all_results[best_key]:
        res = all_results[best_key]['results']
        cm = np.array([[res['tn'], res['fp']], [res['fn'], res['tp']]])
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['Predicted 0', 'Predicted 1'],
                    yticklabels=['Actual 0', 'Actual 1'])
        plt.title(f'Confusion Matrix - {best_key[:30]}')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/Fig6_confusion_matrix.pdf', dpi=300)
        plt.close()

    # Additional figure: Fingerprint count vs AUC
    fp_count_auc = []
    for key, res in all_results.items():
        if 'results' in res:
            auc_val = res['results']['auc']
            fp_count = res.get('fingerprint_count', 1)
            fp_count_auc.append((fp_count, auc_val))
    if fp_count_auc:
        df_count = pd.DataFrame(fp_count_auc, columns=['FP_Count', 'AUC'])
        plt.figure(figsize=(8,6))
        sns.boxplot(x='FP_Count', y='AUC', data=df_count)
        sns.stripplot(x='FP_Count', y='AUC', data=df_count, color='black', size=3, alpha=0.5)
        plt.title('AUC vs Number of Fingerprints Combined')
        plt.xlabel('Number of Fingerprints')
        plt.ylabel('AUC')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/AUC_vs_FPcount.pdf', dpi=300)
        plt.close()
    print(f"\nAll figures saved to {output_dir} directory (PDF format).")

# ==================== Main Program ====================
def main():
    import sys
    sys.argv = [arg for arg in sys.argv if not arg.startswith('--f=')]

    parser = argparse.ArgumentParser(description='Molecular activity prediction model training and plotting')
    parser.add_argument('--plot-only', action='store_true', 
                        help='Only plot based on existing results, no training (requires all_fp_combinations_results.pkl)')
    parser.add_argument('--results-file', type=str, default='all_fp_combinations_results.pkl',
                        help='Results file path (default: all_fp_combinations_results.pkl)')
    args = parser.parse_args()

    print("="*80)
    print("Molecular Activity Prediction Model Training System - Fingerprint Combination Extension")
    print("="*80)

    fp_combinations = generate_all_fp_combinations()
    fingerprint_configs = [
        {'name': 'PubChem', 'type': 'pubchem'},
        {'name': 'Klekota-Roth', 'type': 'klekotaroth'},
        {'name': 'ECFP', 'type': 'ecfp'},
        {'name': 'MACCS', 'type': 'maccs'},
        {'name': 'AtomPairs', 'type': 'atompairs'},
        {'name': 'RDKit', 'type': 'rdkit'},
    ]

    def load_and_split_data():
        try:
            df = pd.read_csv('data.csv')
        except FileNotFoundError:
            print("Error: Cannot find data.csv file")
            return None, None, None, None, None
        print("\nDataset Information:")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Shape: {df.shape}")
        smiles_col = 'SMILES'
        label_col = 'LABEL'
        for col in df.columns:
            if 'smile' in col.lower():
                smiles_col = col
            if 'label' in col.lower() or 'class' in col.lower() or 'activity' in col.lower():
                label_col = col
        print(f"  Using SMILES column: {smiles_col}")
        print(f"  Using label column: {label_col}")
        print("\n=== Data Preprocessing ===")
        valid_smiles = []
        valid_labels = []
        invalid_count = 0
        for idx, smi in enumerate(df[smiles_col]):
            try:
                if pd.isna(smi):
                    continue
                mol = Chem.MolFromSmiles(str(smi))
                if mol is not None:
                    valid_smiles.append(str(smi))
                    valid_labels.append(df[label_col].iloc[idx])
                else:
                    invalid_count += 1
                    if invalid_count <= 5:
                        print(f"  Warning: Invalid SMILES at row {idx}: {smi}")
            except Exception:
                invalid_count += 1
                continue
        if invalid_count > 5:
            print(f"  ... and {invalid_count - 5} other invalid SMILES")
        print(f"\n✓ Valid SMILES count: {len(valid_smiles)}/{len(df)}")
        if len(valid_smiles) == 0:
            print("Error: No valid SMILES found")
            return None, None, None, None, None
        smiles_array = np.array(valid_smiles)
        y = np.array(valid_labels)
        print(f"✓ Label distribution: Class 0: {sum(y==0)}, Class 1: {sum(y==1)}")
        smiles_train, smiles_test, y_train, y_test = train_test_split(
            smiles_array, y, random_state=42, test_size=0.2, stratify=y
        )
        print(f"\nDataset Split:")
        print(f"  Training set: {len(smiles_train)} samples")
        print(f"  Test set: {len(smiles_test)} samples")
        return df, smiles_train, smiles_test, y_train, y_test

    if args.plot_only:
        print("\n[Plot-only Mode]")
        result = load_and_split_data()
        if result[0] is None:
            return
        df, smiles_train, smiles_test, y_train, y_test = result
        all_results = load_all_results(args.results_file)
        if not all_results:
            print(f"Error: Cannot load results file {args.results_file}, please train first.")
            return
        print("\nComputing test set features for plotting...")
        X_test_feats = {}
        for fp_type in all_fp_types:
            pipeline = create_fingerprint_pipeline(fp_type)
            X_test_feats[fp_type] = pipeline.fit_transform(smiles_test).astype(float)
        plot_all_figures(df, all_results, y_test, smiles_test, X_test_feats, fingerprint_configs, fp_combinations)
        print("\nPlotting complete!")
        return

    # Normal training pipeline
    print("\n[6 Molecular Fingerprints]")
    print("  1. PubChem Fingerprint (881 bits)")
    print("  2. Klekota-Roth Fingerprint (4860 bits)")
    print("  3. ECFP Fingerprint (Morgan/ECFP4, 2048 bits)")
    print("  4. MACCS Fingerprint (167 bits)")
    print("  5. AtomPairs Fingerprint (2048 bits)")
    print("  6. RDKit Fingerprint (FP2, 2048 bits)")

    print("\n[Fingerprint Combinations]")
    print(f"  🔹 Single fingerprint: 6 types")
    print(f"  🔹 Dual fingerprint: {len(fp_combinations['2'])} combinations")
    print(f"  🔹 Triple fingerprint: {len(fp_combinations['3'])} combinations")
    print(f"  🔹 Quadruple fingerprint: {len(fp_combinations['4'])} combinations")
    print(f"  🔹 Quintuple fingerprint: {len(fp_combinations['5'])} combinations")
    print(f"  🔹 Sextuple fingerprint: 1 combination")
    print(f"  🔹 Total fingerprint combinations: {6 + len(fp_combinations['2']) + len(fp_combinations['3']) + len(fp_combinations['4']) + len(fp_combinations['5']) + 1}")

    print("\n[All Models]")
    print("  🔹 5 base models (for single fingerprints)")
    print("  🔹 2 ensemble models (for single fingerprints)")
    print("  🔹 3 multi-fingerprint ensemble models (for all fingerprint combinations)")

    total_combinations = (
        6 * 5 +
        6 * 2 +
        (len(fp_combinations['2']) + len(fp_combinations['3']) + 
         len(fp_combinations['4']) + len(fp_combinations['5']) + 1) * 3
    )

    print(f"\n[Training Combinations]")
    print(f"  Single FP models: 6 fingerprints × (5 base + 2 ensemble) = 42 combinations")
    print(f"  Multi-FP models: {len(fp_combinations['2'])+len(fp_combinations['3'])+len(fp_combinations['4'])+len(fp_combinations['5'])+1} FP combinations × 3 models = {(len(fp_combinations['2'])+len(fp_combinations['3'])+len(fp_combinations['4'])+len(fp_combinations['5'])+1)*3} combinations")
    print(f"  Total: {total_combinations} model-fingerprint combinations")
    print("="*80)

    checkpoint_dir = 'all_fp_combinations_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"\n✓ Checkpoint directory: {checkpoint_dir}")

    data_result = load_and_split_data()
    if data_result[0] is None:
        return
    df, smiles_train, smiles_test, y_train, y_test = data_result

    base_models_to_train = {
        'KNN': train_knn,
        'SVM': train_svm,
        'RandomForest': train_random_forest,
        'XGBoost': train_xgboost,
        'GaussianNB': train_gaussian_nb
    }
    ensemble_models_to_train = {
        'VotingEnsemble': train_voting_ensemble,
        'StackingEnsemble': train_stacking_ensemble
    }
    multi_fp_models_to_train = {
        'UnionPipeline': train_multi_fp_union_pipeline,
        'Voting': train_multi_fp_voting,
        'Stacking': train_multi_fp_stacking
    }

    all_results = load_all_results('all_fp_combinations_results.pkl')

    # Stage 1: Single fingerprint base models
    print(f"\n{'#'*80}")
    print("# Stage 1: Training single fingerprint base models (6 fingerprints × 5 models = 30 combinations)")
    print(f"{'#'*80}")
    total_combinations_stage1 = len(fingerprint_configs) * len(base_models_to_train)
    current_combination = 0
    for fp_idx, fp_config in enumerate(fingerprint_configs, 1):
        fp_name = fp_config['name']
        fp_type = fp_config['type']
        print(f"\n{'#'*80}")
        print(f"# Fingerprint [{fp_idx}/{len(fingerprint_configs)}]: {fp_name}")
        print(f"{'#'*80}")
        for model_idx, (model_name, train_func) in enumerate(base_models_to_train.items(), 1):
            current_combination += 1
            print(f"\n{'-'*60}")
            print(f"[{current_combination}/{total_combinations_stage1}] Processing: {model_name} × {fp_name}")
            print(f"{'-'*60}")
            checkpoint_key = f"{model_name}_{fp_type}"
            checkpoint_name = f"{fp_name}_{model_name}"
            if is_model_trained(checkpoint_name, model_name, checkpoint_dir):
                print(f"  ✓ Model already trained, loading checkpoint...")
                checkpoint = load_checkpoint(checkpoint_name, model_name, checkpoint_dir)
                if checkpoint:
                    all_results[checkpoint_key] = checkpoint
                continue
            try:
                result = train_func(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type)
                model_result = {
                    'model_name': model_name,
                    'fp_name': fp_name,
                    'fp_type': fp_type,
                    'results': result['results'],
                    'model': result.get('model'),
                    'fp_pipeline': result.get('fp_pipeline'),
                    'scaler': result.get('scaler'),
                    'best_params': result['best_params'],
                    'cv_score': result['cv_score'],
                    'fingerprint_count': 1,
                    'fingerprint_list': [fp_type]
                }
                all_results[checkpoint_key] = model_result
                save_checkpoint(checkpoint_name, model_name, model_result, checkpoint_dir)
                save_all_results(all_results, 'all_fp_combinations_results.pkl')
            except Exception as e:
                print(f"  ✗ Error training {model_name} × {fp_name}: {e}")
                import traceback
                traceback.print_exc()
                continue

    # Stage 2: Single fingerprint ensemble models
    print(f"\n{'#'*80}")
    print("# Stage 2: Training single fingerprint ensemble models (6 fingerprints × 2 models = 12 combinations)")
    print(f"{'#'*80}")
    total_combinations_stage2 = len(fingerprint_configs) * len(ensemble_models_to_train)
    current_combination = 0
    for fp_idx, fp_config in enumerate(fingerprint_configs, 1):
        fp_name = fp_config['name']
        fp_type = fp_config['type']
        for model_idx, (model_name, train_func) in enumerate(ensemble_models_to_train.items(), 1):
            current_combination += 1
            checkpoint_key = f"{model_name}_{fp_type}"
            checkpoint_name = f"{fp_name}_{model_name}"
            print(f"\n{'-'*60}")
            print(f"[{current_combination}/{total_combinations_stage2}] Processing: {model_name} × {fp_name}")
            print(f"{'-'*60}")
            if is_model_trained(checkpoint_name, model_name, checkpoint_dir):
                print(f"  ✓ Model already trained, loading checkpoint...")
                checkpoint = load_checkpoint(checkpoint_name, model_name, checkpoint_dir)
                if checkpoint:
                    all_results[checkpoint_key] = checkpoint
                continue
            try:
                result = train_func(smiles_train, smiles_test, y_train, y_test, fp_name, fp_type)
                model_result = {
                    'model_name': model_name,
                    'fp_name': fp_name,
                    'fp_type': fp_type,
                    'results': result['results'],
                    'model': result.get('model'),
                    'fp_pipeline': result.get('fp_pipeline'),
                    'best_params': result['best_params'],
                    'cv_score': result['cv_score'],
                    'fingerprint_count': 1,
                    'fingerprint_list': [fp_type]
                }
                all_results[checkpoint_key] = model_result
                save_checkpoint(checkpoint_name, model_name, model_result, checkpoint_dir)
                save_all_results(all_results, 'all_fp_combinations_results.pkl')
            except Exception as e:
                print(f"  ✗ Error training {model_name} × {fp_name}: {e}")
                continue

    # Stage 3: Multi-fingerprint combination ensemble models
    print(f"\n{'#'*80}")
    print("# Stage 3: Training multi-fingerprint combination ensemble models")
    print(f"  Dual fingerprints: {len(fp_combinations['2'])} combinations")
    print(f"  Triple fingerprints: {len(fp_combinations['3'])} combinations")
    print(f"  Quadruple fingerprints: {len(fp_combinations['4'])} combinations")
    print(f"  Quintuple fingerprints: {len(fp_combinations['5'])} combinations")
    print(f"  Sextuple fingerprints: 1 combination")
    print(f"  Total: {len(fp_combinations['2'])+len(fp_combinations['3'])+len(fp_combinations['4'])+len(fp_combinations['5'])+1} fingerprint combinations × 3 models")
    print(f"{'#'*80}")
    total_combinations_stage3 = (len(fp_combinations['2']) + len(fp_combinations['3']) + 
                                 len(fp_combinations['4']) + len(fp_combinations['5']) + 1) * len(multi_fp_models_to_train)
    current_combination = 0
    for combo_size in ['2', '3', '4', '5', '6']:
        combinations = fp_combinations[combo_size]
        size_name = f"{combo_size} fingerprints" if combo_size != '6' else "six fingerprints"
        print(f"\n{'#'*80}")
        print(f"# Processing {size_name} combinations (total {len(combinations)})")
        print(f"{'#'*80}")
        for combo_idx, fp_combo in enumerate(combinations, 1):
            fp_types = list(fp_combo)
            fp_names = [fp_type_to_name[ft] for ft in fp_types]
            combo_name = f"{size_name}_{'+'.join([n[:3] for n in fp_names])}"
            print(f"\n{'='*70}")
            print(f"[{combo_idx}/{len(combinations)}] Fingerprint combination: {combo_name}")
            print(f"{'='*70}")
            for model_idx, (model_short_name, train_func) in enumerate(multi_fp_models_to_train.items(), 1):
                model_full_name = f"MultiFP{model_short_name}"
                current_combination += 1
                checkpoint_key = f"{model_full_name}_{combo_name}"
                checkpoint_name = f"{combo_name}_{model_full_name}"
                print(f"\n{'-'*60}")
                print(f"  [{current_combination}/{total_combinations_stage3}] Model: {model_full_name}")
                print(f"{'-'*60}")
                if is_model_trained(checkpoint_name, model_full_name, checkpoint_dir):
                    print(f"  ✓ Model already trained, loading checkpoint...")
                    checkpoint = load_checkpoint(checkpoint_name, model_full_name, checkpoint_dir)
                    if checkpoint:
                        all_results[checkpoint_key] = checkpoint
                    continue
                try:
                    result = train_func(
                        smiles_train, smiles_test, y_train, y_test, 
                        fp_types, combo_name
                    )
                    model_result = {
                        'model_name': model_full_name,
                        'fp_name': combo_name,
                        'fp_type': 'multi_fp',
                        'results': result['results'],
                        'model': result['model'],
                        'best_params': result['best_params'],
                        'cv_score': result['cv_score'],
                        'fingerprint_count': len(fp_types),
                        'fingerprint_list': fp_types,
                        'fingerprint_names': fp_names
                    }
                    all_results[checkpoint_key] = model_result
                    save_checkpoint(checkpoint_name, model_full_name, model_result, checkpoint_dir)
                    save_all_results(all_results, 'all_fp_combinations_results.pkl')
                except Exception as e:
                    print(f"  ✗ Error training {model_full_name} × {combo_name}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

    # Performance comparison
    print(f"\n{'='*100}")
    print("[Final Results] All Fingerprint Combinations × All Models - Comprehensive Performance Comparison")
    print(f"{'='*100}")
    comparison_data = []
    for key, result in all_results.items():
        if 'results' in result and result['results']['auc'] > 0.5:
            comparison_data.append({
                'Model': result['model_name'],
                'Fingerprint': result['fp_name'],
                'FP_Count': result.get('fingerprint_count', 1),
                'AUC': result['results']['auc'],
                'Accuracy': result['results']['accuracy'],
                'F1': result['results']['f1'],
                'MCC': result['results']['mcc'],
                'Sensitivity': result['results']['se'],
                'Specificity': result['results']['sp'],
                'BA': result['results']['ba']
            })
    if comparison_data:
        results_df = pd.DataFrame(comparison_data)
        results_df = results_df.sort_values('AUC', ascending=False)
        print(f"\n{'Rank':<4} {'Model':<20} {'Fingerprint Combination':<30} {'FP Count':<8} {'AUC':<8} {'MCC':<8}")
        print(f"{'-'*100}")
        for i, (_, row) in enumerate(results_df.head(20).iterrows(), 1):
            fp_name = row['Fingerprint'][:28] + '..' if len(row['Fingerprint']) > 28 else row['Fingerprint']
            print(f"{i:<4} {row['Model']:<20} {fp_name:<30} {row['FP_Count']:<8} {row['AUC']:.4f}    {row['MCC']:.4f}")
        results_df.to_csv('all_fp_combinations_comparison.csv', index=False)
        print(f"\n✓ Complete performance comparison saved to: all_fp_combinations_comparison.csv")

        # Confusion matrix table
        confusion_data = []
        for key, result in all_results.items():
            if 'results' in result:
                confusion_data.append({
                    'Model_Key': key,
                    'Model_Name': result.get('model_name', ''),
                    'Fingerprint': result.get('fp_name', ''),
                    'FP_Count': result.get('fingerprint_count', 1),
                    'TP': result['results']['tp'],
                    'TN': result['results']['tn'],
                    'FP': result['results']['fp'],
                    'FN': result['results']['fn'],
                    'Total_Samples': result['results']['tp'] + result['results']['tn'] + result['results']['fp'] + result['results']['fn']
                })
        confusion_df = pd.DataFrame(confusion_data)
        confusion_df.to_csv('all_fp_combinations_confusion.csv', index=False)
        print(f"✓ Confusion matrices for all models saved to: all_fp_combinations_confusion.csv")

        for fp_count in [1, 2, 3, 4, 5, 6]:
            count_results = results_df[results_df['FP_Count'] == fp_count]
            if not count_results.empty:
                best = count_results.iloc[0]
                size_name = f"{fp_count} fingerprints" if fp_count < 6 else "six fingerprints"
                print(f"\n{size_name}:")
                print(f"  Best combination: {best['Model']} + {best['Fingerprint']}")
                print(f"  AUC: {best['AUC']:.4f}, MCC: {best['MCC']:.4f}")

        best_model_key = max(all_results.keys(), 
                           key=lambda k: all_results[k]['results']['auc'] 
                           if 'results' in all_results[k] else 0)
        best_result = all_results[best_model_key]
        print(f"\n{'='*100}")
        print("🏆 Overall Best Model")
        print(f"{'='*100}")
        print(f"  Model: {best_result['model_name']}")
        print(f"  Fingerprint combination: {best_result['fp_name']}")
        print(f"  Fingerprint count: {best_result.get('fingerprint_count', 1)}")
        print(f"  AUC: {best_result['results']['auc']:.4f}")
        print(f"  MCC: {best_result['results']['mcc']:.4f}")
        print(f"  F1:  {best_result['results']['f1']:.4f}")
        print(f"  Sensitivity: {best_result['results']['se']:.4f}")
        print(f"  Specificity: {best_result['results']['sp']:.4f}")

        model_filename = f"BEST_MODEL_{best_result['model_name']}_{best_result['fp_name'].replace(' ', '_')[:30]}_AUC{best_result['results']['auc']:.4f}.pkl"
        joblib.dump({
            'model': best_result['model'],
            'model_name': best_result['model_name'],
            'fingerprint_name': best_result['fp_name'],
            'fingerprint_count': best_result.get('fingerprint_count', 1),
            'fingerprint_list': best_result.get('fingerprint_list', []),
            'best_params': best_result['best_params'],
            'performance': best_result['results'],
            'all_results': results_df.to_dict('records')
        }, model_filename)
        print(f"\n✓ Best model saved as: {model_filename}")

        # Test prediction
        test_smiles = smiles_test[:min(3, len(smiles_test))]
        print(f"\n{'='*100}")
        print("Prediction Function Test - Using Best Model")
        print(f"{'='*100}")
        print(f"\nTest SMILES:")
        for i, smi in enumerate(test_smiles):
            print(f"  {i+1}: {smi[:50]}...")
        try:
            if 'Voting' in best_result['model_name'] and best_result.get('fingerprint_count', 1) > 1:
                prediction_result = predict_activity(
                    test_smiles,
                    best_result['model'],
                    model_type='multi_fp_voting'
                )
            elif best_result.get('fingerprint_count', 1) > 1:
                prediction_result = predict_activity(
                    test_smiles,
                    best_result['model'],
                    model_type='pipeline'
                )
            else:
                prediction_result = predict_activity(
                    test_smiles,
                    best_result,
                    model_type='single'
                )
            print(f"\nPrediction Results:")
            for i, (smi, activity, prob) in enumerate(zip(
                prediction_result['smiles'], 
                prediction_result['activity'], 
                prediction_result['probabilities']
            )):
                print(f"  {i+1}: {smi[:50]}... -> {activity} (Probability: {prob:.4f})")
        except Exception as e:
            print(f"Error during prediction: {e}")
            import traceback
            traceback.print_exc()

        # External file prediction
        external_file = "TZJ179.csv"
        if os.path.exists(external_file):
            try:
                ext_df = pd.read_csv(external_file)
                smiles_col = None
                for col in ext_df.columns:
                    if 'smile' in col.lower():
                        smiles_col = col
                        break
                if smiles_col is None:
                    print("Error: Cannot find SMILES column in TZJ179.csv")
                else:
                    ext_smiles = ext_df[smiles_col].astype(str).tolist()
                    print(f"Read {len(ext_smiles)} SMILES")
                    if 'Voting' in best_result['model_name'] and best_result.get('fingerprint_count', 1) > 1:
                        pred_result = predict_activity(
                            ext_smiles,
                            best_result['model'],
                            model_type='multi_fp_voting'
                        )
                    elif best_result.get('fingerprint_count', 1) > 1:
                        pred_result = predict_activity(
                            ext_smiles,
                            best_result['model'],
                            model_type='pipeline'
                        )
                    else:
                        pred_result = predict_activity(
                            ext_smiles,
                            best_result,
                            model_type='single'
                        )
                    print("\nPrediction Results:")
                    for i, (smi, activity, prob) in enumerate(zip(
                        pred_result['smiles'], 
                        pred_result['activity'], 
                        pred_result['probabilities']
                    )):
                        print(f"  {i+1}: {smi[:60]}... -> {activity} (Probability: {prob:.4f})")
                    output_df = pd.DataFrame({
                        'SMILES': pred_result['smiles'],
                        'Predicted_Activity': pred_result['activity'],
                        'Probability': pred_result['probabilities']
                    })
                    output_file = "TZJ179_predictions.csv"
                    output_df.to_csv(output_file, index=False)
                    print(f"\nPrediction results saved to: {output_file}")
            except Exception as e:
                print(f"Error predicting external file: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"File {external_file} does not exist, skipping external prediction.")

    # Plotting
    print("\nPreparing test set features for plotting...")
    X_test_feats = {}
    for fp_type in all_fp_types:
        pipeline = create_fingerprint_pipeline(fp_type)
        X_test_feats[fp_type] = pipeline.fit_transform(smiles_test).astype(float)

    plot_all_figures(df, all_results, y_test, smiles_test, X_test_feats, fingerprint_configs, fp_combinations)

    print(f"\n{'='*100}")
    print("✓ All training completed!")
    print(f"  Total models trained: {len(all_results)}")
    print(f"  Effective models: {len(comparison_data)}")
    print(f"  Fingerprint combinations: 6 single + {len(fp_combinations['2'])} dual + {len(fp_combinations['3'])} triple + {len(fp_combinations['4'])} quadruple + {len(fp_combinations['5'])} quintuple + 1 sextuple = {6+len(fp_combinations['2'])+len(fp_combinations['3'])+len(fp_combinations['4'])+len(fp_combinations['5'])+1} types")
    print(f"{'='*100}")

if __name__ == "__main__":
    main()